# Finsheild Phase 1 — Dataset pipeline (Colab-ready)
Reproducible loader / preprocessing / splits on the Kaggle Credit Card Fraud dataset. No LLM weights. CPU-only, but runs on GPU runtime as well.

In [ ]:
# Cell 1 — Hardware detection (python, torch, CUDA, GPU mem)
import sys, platform, torch
print(f"Python {sys.version}")
print(f"Platform {platform.platform()}")
print(f"Torch {torch.__version__ if 'torch' in sys.modules else 'not installed yet'}")
try:
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        print(f"GPU mem: {props.total_memory/1e9:.1f} GB")
    else:
        print("Running CPU-only (expected for Phase 1)")
except Exception as e:
    print(f"Torch check skipped: {e}")
    print("Running CPU-only")


In [ ]:
# Cell 2 — Install deps (Colab-friendly)
!pip install -r requirements-colab.txt  2>&1 | tail -n 20
# Fallback if file not found when running from different cwd:
# !pip install pandas scikit-learn pyyaml matplotlib seaborn joblib kagglehub opendatasets tqdm


In [ ]:
# Cell 3 — Download dataset (Kaggle primary, synthetic fallback)
# Option A: KaggleHub (requires KAGGLE_USERNAME/KAGGLE_KEY or kaggle.json — set as Colab Secrets)
import os, pathlib
from pathlib import Path
print("Attempting KaggleHub download...")
try:
    import kagglehub
    dest = Path("data/raw/creditcard.csv")
    if not dest.exists():
        cache = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        import shutil
        for p in Path(cache).rglob("creditcard.csv"):
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(p, dest)
            print(f"Downloaded to {dest} via kagglehub")
            break
    else:
        print(f"Already exists at {dest}")
except Exception as e:
    print(f"KaggleHub failed: {e}")
    print("Fallback: run synthetic — !python scripts/download_dataset.py --synthetic")
    # Synthetic fallback for Colab without creds:
    # !python scripts/download_dataset.py --synthetic
    print("Or set Colab Secrets: KAGGLE_USERNAME / KAGGLE_KEY and restart runtime")


In [ ]:
# Cell 4 — Alternative download via opendatasets (also needs Kaggle creds)
# !pip install opendatasets -q
# import opendatasets as od
# od.download('https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud', data_dir='data/raw/_tmp')
# Direct URL fallback (if dataset is mirrored): document in docs/dataset.md


In [ ]:
# Cell 5 — Loader / preprocessing / splits demo (must NOT leak)
from finsheild.data.loader import load_raw
from finsheild.data.splits import make_splits, save_splits
from finsheild.data.preprocessing import FraudPreprocessor, preprocess_splits
import pandas as pd

df = load_raw("data/raw/creditcard.csv")
print(df.head())
print(df["Class"].value_counts())

train, val, test = make_splits(df, test_size=0.15, val_size=0.15, random_state=42)
print(f"\nSplits: train {train.shape}, val {val.shape}, test {test.shape}")

# Leakage-safe preprocessing: fit ONLY on train
pre = FraudPreprocessor(scale_features=["Amount", "Time"])
train_t = pre.fit_transform_train(train)
val_t = pre.transform(val)
test_t = pre.transform(test)
print("\nScaler means:", pre.scaler.mean_)
print("Train Amount mean after scaling:", train_t["Amount"].mean())

# Save processed + scaler (gitignored)
save_splits(train_t, val_t, test_t, out_dir="data/processed", fmt="csv")
pre.save("data/processed/scaler.joblib")
print("Saved processed splits + scaler")


In [ ]:
# Cell 6 — EDA quick checks (shape, dtypes, missing, class dist, Amount/Time by class)
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
print(f"Shape: {df.shape}")
print(df.dtypes)
print(f"Missing: {df.isnull().sum().sum() == 0}")
print(f"Duplicated rows: {df.duplicated().sum()}")
print(df["Class"].value_counts(normalize=True))

Path("evaluation/figures").mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(4,3))
sns.countplot(x="Class", data=df)
plt.title("Class distribution")
plt.savefig("evaluation/figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1,2, figsize=(10,4))
for ax, col in zip(axes, ["Amount","Time"]):
    df.boxplot(column=col, by="Class", ax=ax)
    ax.set_title(col)
plt.suptitle("")
plt.tight_layout()
plt.savefig("evaluation/figures/amount_time_by_class.png", dpi=150, bbox_inches="tight")
plt.show()
